In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

THU_MUC_GOC = Path("..").resolve()
THU_MUC_DAC_TRUNG = THU_MUC_GOC / "data" / "features"
THU_MUC_MODEL = THU_MUC_GOC / "model"
THU_MUC_REPORT = THU_MUC_GOC / "report"

THU_MUC_MODEL.mkdir(parents=True, exist_ok=True)
THU_MUC_REPORT.mkdir(parents=True, exist_ok=True)

X_train = np.load(THU_MUC_DAC_TRUNG / "X_train.npy")
y_train = np.load(THU_MUC_DAC_TRUNG / "y_train.npy")
print(f"[SVM] Đã nạp X_train: {X_train.shape} | y_train: {y_train.shape}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

bat_dau = time.time()

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(class_weight="balanced", cache_size=500, random_state=42)),
])

param_svm = [
    {"clf__kernel": ["rbf"], "clf__C": [1, 5, 10], "clf__gamma": ["scale"]},
    {"clf__kernel": ["linear"], "clf__C": [0.1, 1]},
]

grid_svm = GridSearchCV(
    pipe_svm,
    param_grid=param_svm,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
)
grid_svm.fit(X_train, y_train)

print(f"\nTìm xong tham số tốt nhất ({grid_svm.best_params_})")

best_p = grid_svm.best_params_
svm_final = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel=best_p["clf__kernel"],
        C=best_p["clf__C"],
        gamma=best_p.get("clf__gamma", "scale"),
        class_weight="balanced",
        probability=True,
        cache_size=500,
        random_state=42,
    )),
])
svm_final.fit(X_train, y_train)

thoi_gian = time.time() - bat_dau
joblib.dump(svm_final, THU_MUC_MODEL / "svm_model.pkl")

ket_qua = {
    "model": "SVM",
    "best_params": best_p,
    "cv_f1_macro": round(float(grid_svm.best_score_), 4),
    "train_time_sec": round(thoi_gian, 1),
}

with open(THU_MUC_REPORT / "cv_svm.json", "w", encoding="utf-8") as f:
    json.dump(ket_qua, f, ensure_ascii=False, indent=2)

print("\nHoàn tất SVM:", ket_qua)

[SVM] Đã nạp X_train: (12888, 1280) | y_train: (12888,)
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Tìm xong tham số tốt nhất ({'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}) — đang huấn luyện lại bản có probability=True...


c:\Users\Nguyen Vu Thuan\Documents\ML_bao_cao\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



Hoàn tất SVM: {'model': 'SVM', 'best_params': {'clf__C': 10, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}, 'cv_f1_macro': 0.9657, 'train_time_sec': 599.2}
